# Information Theory — Exercises

Companion to the note [Information Theory](Information%20Theory.md).

Practise entropy, cross-entropy, KL divergence (and its asymmetry), mutual information, information gain for decision-tree splits and the Jensen–Shannon divergence: by hand, then in NumPy, checked against SciPy and scikit-learn.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×7 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.stats import entropy as sp_entropy
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import log_loss, mutual_info_score
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
rng = np.random.default_rng(0)

## Part A · Concepts

### Exercise 1 · Entropy as uncertainty
*🧠 Concept · ★☆☆*

(a) Among all distributions over $K$ outcomes, which has the largest entropy and what is its value? Which has the smallest?
(b) What is the difference between measuring entropy in bits and in nats?
(c) Why does a decision tree prefer splits that produce low-entropy children ([[Decision Trees]])?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Entropy is the expected surprise $-\log p(x)$. Base 2 counts yes/no questions.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) The uniform distribution, $H=\log K$; a point mass (one outcome with probability 1) has $H=0$.
(b) Only the log base: $H_{\text{bits}} = H_{\text{nats}}/\ln 2$. Bits = average number of optimal yes/no questions (or code bits).
(c) Low child entropy means nearly pure leaves: after the split we are almost certain of the class, so the split removed a lot
of uncertainty (high information gain).

</details>

### Exercise 2 · Forward versus reverse KL
*🧠 Concept · ★★☆*

Let $p$ be a bimodal distribution and fit a single Gaussian $q$ to it by minimising either (i) $D_{KL}(p\Vert q)$ or (ii)
$D_{KL}(q\Vert p)$. Which objective makes $q$ cover both modes and which makes it lock onto one mode? Which of the two is
used in maximum likelihood, and which in variational inference ([[Variational Inference]])?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$D_{KL}(p\Vert q) = \sum p\log\frac pq$ explodes where $p>0$ but $q\approx 0$. $D_{KL}(q\Vert p)$ explodes where $q>0$ but $p\approx0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(i) **Forward KL** $D_{KL}(p\Vert q)$ heavily penalises $q\approx0$ where $p$ has mass, so $q$ spreads to cover both modes
(mass-covering, possibly putting mass between the modes). (ii) **Reverse KL** $D_{KL}(q\Vert p)$ penalises $q$ having mass where
$p\approx0$, so $q$ sits on one mode (mode-seeking).
MLE minimises forward KL from the data distribution to the model (Exercise 3). VI maximises the ELBO, which equals minimising
reverse KL $D_{KL}(q\Vert p(\cdot\mid x))$, hence its known tendency to underestimate posterior variance.

</details>

### Exercise 3 · Cross-entropy = entropy + KL
*🧠 Concept · ★☆☆*

Show that $H(p,q) = H(p) + D_{KL}(p\Vert q)$. Use it to explain the note's claim "minimising cross-entropy = minimising KL to the
data distribution", and why a classifier's cross-entropy loss can never go below $H(p)$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Write $\log\frac pq = \log p - \log q$ inside the KL sum.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$D_{KL}(p\Vert q) = \sum p\log p - \sum p\log q = -H(p) + H(p,q)$. Rearranged: $H(p,q) = H(p)+D_{KL}(p\Vert q)$.
$H(p)$ does not depend on the model $q$, so $\arg\min_q H(p,q) = \arg\min_q D_{KL}(p\Vert q)$. Since $D_{KL}\ge0$, the loss is
bounded below by $H(p)$, the irreducible label noise; for one-hot labels $H(p)=0$.

</details>

### Exercise 4 · Mutual information versus correlation
*🧠 Concept · ★★☆*

Give an example of variables $X, Y$ with zero (Pearson) correlation but large mutual information. Why does this make mutual
information attractive for feature selection ([[Feature Engineering]])? Mention one practical drawback.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Try $Y = X^2$ with $X$ symmetric around 0.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$X\sim\mathcal N(0,1)$, $Y=X^2$: $\mathrm{Cov}(X,Y)=\mathbb E[X^3]=0$, yet $Y$ is a deterministic function of $X$, so
$I(X;Y)$ is large ($H(Y\mid X)=0$). MI is zero **iff** $X\perp Y$, so it detects any (non-linear) dependence, while correlation only
measures linear association. Drawback: MI of continuous variables must be *estimated* (binning, k-NN estimators), which is noisy,
biased for small samples and sensitive to hyperparameters; and univariate MI ignores redundancy between features.

</details>

## Part B · By hand

### Exercise 5 · Entropy of a biased coin
*✍️ By hand · ★☆☆*

Compute the entropy in **bits** of a coin with $p(\text{heads})=0.25$, and of a fair coin. Which is more predictable?

In [ ]:
H_biased = None
H_fair = None

check('H(0.25) in bits', H_biased, sp_entropy([0.25, 0.75], base=2))
check('H(0.5) in bits', H_fair, sp_entropy([0.5, 0.5], base=2))

<details>
<summary><b>💡 Hint</b></summary>

$\log_2 0.25 = -2$ and $\log_2 0.75\approx-0.415$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$H = -0.25\log_2 0.25 - 0.75\log_2 0.75 = 0.5 + 0.311 = 0.811$ bits; a fair coin has $1$ bit. The biased coin is more predictable.

```python
H_biased = 0.8112781
H_fair = 1.0
```

</details>

### Exercise 6 · Cross-entropy loss for one prediction
*✍️ By hand · ★☆☆*

A 3-class classifier outputs $q=(0.2, 0.7, 0.1)$ and the true class is the second one (one-hot $p=(0,1,0)$). Compute the
cross-entropy loss in nats. What would it be if the model had put $0.01$ on the true class?

In [ ]:
ce_07 = None
ce_001 = None

check('CE with q=0.7', ce_07, log_loss([1], [[0.2, 0.7, 0.1]], labels=[0, 1, 2]))
check('CE with q=0.01', ce_001, -np.log(0.01))

<details>
<summary><b>💡 Hint</b></summary>

With a one-hot $p$, $H(p,q) = -\log q_{\text{true}}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$-\ln 0.7\approx0.357$; $-\ln0.01\approx4.605$. Confident mistakes are punished very heavily (the loss is unbounded as $q_{\text{true}}\to0$).

```python
ce_07 = -np.log(0.7)
ce_001 = -np.log(0.01)
```

</details>

### Exercise 7 · KL divergence is asymmetric
*✍️ By hand · ★★☆*

For $p=(0.5, 0.5)$ and $q=(0.9, 0.1)$ compute $D_{KL}(p\Vert q)$ and $D_{KL}(q\Vert p)$ in nats.

In [ ]:
kl_pq = None
kl_qp = None

check('KL(p||q)', kl_pq, sp_entropy([0.5, 0.5], [0.9, 0.1]))
check('KL(q||p)', kl_qp, sp_entropy([0.9, 0.1], [0.5, 0.5]))

<details>
<summary><b>💡 Hint</b></summary>

$D_{KL}(p\Vert q) = 0.5\ln\frac{0.5}{0.9} + 0.5\ln\frac{0.5}{0.1}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$D_{KL}(p\Vert q) = 0.5(-0.5878) + 0.5(1.6094) \approx 0.511$; $D_{KL}(q\Vert p) = 0.9\ln1.8 + 0.1\ln0.2 = 0.5290 - 0.1609\approx 0.368$.
Different values, so KL is not a distance (it also violates the triangle inequality).

```python
kl_pq = 0.5 * np.log(0.5 / 0.9) + 0.5 * np.log(0.5 / 0.1)
kl_qp = 0.9 * np.log(0.9 / 0.5) + 0.1 * np.log(0.1 / 0.5)
```

</details>

### Exercise 8 · Information gain of a split
*✍️ By hand · ★★☆*

A node has 10 samples, 5 positive and 5 negative. A split sends (4+, 1−) left and (1+, 4−) right. Compute the information gain
in bits using the note's formula $\text{IG} = H(\text{parent}) - \sum_k\frac{n_k}{n}H(\text{child}_k)$. A second split gives
(5+, 3−) and (0+, 2−): which split is better?

In [ ]:
ig_split1 = None
ig_split2 = None

_H = lambda c: sp_entropy(c, base=2)
check('IG split 1', ig_split1, _H([5, 5]) - 0.5 * _H([4, 1]) - 0.5 * _H([1, 4]))
check('IG split 2', ig_split2, _H([5, 5]) - 0.8 * _H([5, 3]) - 0.2 * _H([0, 2]))

<details>
<summary><b>💡 Hint</b></summary>

$H(0.8, 0.2) = -0.8\log_2 0.8 - 0.2\log_2 0.2\approx0.722$ bits; $H(5/8, 3/8)\approx 0.954$ bits; a pure node has $H=0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Split 1: $1 - (0.5\cdot0.722 + 0.5\cdot0.722) = 0.278$ bits. Split 2: $1 - (0.8\cdot0.954 + 0.2\cdot0) = 1-0.764 = 0.236$ bits.
Split 1 is better even though split 2 creates a pure leaf, because its big child stays almost as mixed as the parent.

```python
_h = lambda p: -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
ig_split1 = 1 - _h(0.8)
ig_split2 = 1 - 0.8 * _h(5 / 8)
```

</details>

### Exercise 9 · Mutual information from a joint table
*✍️ By hand · ★★☆*

For the joint distribution $p(x,y)$ with rows $x\in\{0,1\}$ and columns $y\in\{0,1\}$:
$\begin{pmatrix}0.3&0.2\\0.1&0.4\end{pmatrix}$, compute $H(X)$, $H(Y)$, $H(X,Y)$ and $I(X;Y) = H(X)+H(Y)-H(X,Y)$ in bits.
Check that it equals $H(X)-H(X\mid Y)$ (the note's definition).

In [ ]:
P_xy = np.array([[0.3, 0.2], [0.1, 0.4]])
mi_bits = None

_px, _py = P_xy.sum(1), P_xy.sum(0)
check('I(X;Y) in bits', mi_bits, np.sum(P_xy * np.log2(P_xy / np.outer(_px, _py))))

<details>
<summary><b>💡 Hint</b></summary>

Marginals: $p(x)=(0.5,0.5)$, $p(y)=(0.4,0.6)$. $H(X,Y)$ is the entropy of the four joint probabilities.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$H(X)=1$, $H(Y)=H(0.4)\approx0.971$, $H(X,Y) = -\sum p\log_2p \approx 1.846$, so $I\approx 1 + 0.971 - 1.846 = 0.125$ bits.
Equivalently $H(X\mid Y) = H(X,Y)-H(Y) \approx 0.875$ and $I = H(X) - H(X\mid Y) = 0.125$. Knowing $Y$ removes only about an
eighth of a bit of uncertainty about $X$.

```python
P_xy = np.array([[0.3, 0.2], [0.1, 0.4]])
_Hb = lambda p: -np.sum(p * np.log2(p))
mi_bits = _Hb(P_xy.sum(1)) + _Hb(P_xy.sum(0)) - _Hb(P_xy.ravel())
```

</details>

### Exercise 10 · KL between two Gaussians
*✍️ By hand · ★★★*

Derive
$$D_{KL}\big(\mathcal N(\mu_1,\sigma_1^2)\,\Vert\,\mathcal N(\mu_2,\sigma_2^2)\big) = \ln\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1-\mu_2)^2}{2\sigma_2^2} - \frac12$$
and evaluate it for $\mathcal N(0,1)\Vert\mathcal N(1,4)$. Special case: write the VAE regulariser
$D_{KL}(\mathcal N(\mu,\sigma^2)\Vert\mathcal N(0,1))$ ([[Generative Models]]).

In [ ]:
kl_gauss = None

from scipy import stats as _st
from scipy.integrate import quad
_p, _q = _st.norm(0, 1), _st.norm(1, 2)
check('KL(N(0,1) || N(1,4))', kl_gauss, quad(lambda x: _p.pdf(x) * (_p.logpdf(x) - _q.logpdf(x)), -12, 12)[0])

<details>
<summary><b>💡 Hint</b></summary>

$D_{KL} = \mathbb E_p[\log p - \log q]$. Use $\mathbb E_p[(x-\mu_1)^2]=\sigma_1^2$ and $\mathbb E_p[(x-\mu_2)^2]=\sigma_1^2+(\mu_1-\mu_2)^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log p(x)-\log q(x) = \ln\frac{\sigma_2}{\sigma_1} - \frac{(x-\mu_1)^2}{2\sigma_1^2} + \frac{(x-\mu_2)^2}{2\sigma_2^2}$. Taking $\mathbb E_p$:
$\ln\frac{\sigma_2}{\sigma_1} - \frac12 + \frac{\sigma_1^2+(\mu_1-\mu_2)^2}{2\sigma_2^2}$. ✔
With $\mu_1=0,\sigma_1=1,\mu_2=1,\sigma_2=2$: $\ln 2 + \frac{1+1}{8} - \frac12 \approx 0.693+0.25-0.5 = 0.443$.
VAE: $D_{KL}(\mathcal N(\mu,\sigma^2)\Vert\mathcal N(0,1)) = \frac12(\mu^2+\sigma^2-1-\ln\sigma^2)$.

```python
kl_gauss = np.log(2) + 2 / 8 - 0.5
```

</details>

### Exercise 11 · Gibbs' inequality
*✍️ By hand · ★★★*

Prove $D_{KL}(p\Vert q)\ge0$ with equality iff $p=q$, using Jensen's inequality for the concave $\log$.
Conclude that $H(p)\le\log K$ for any distribution over $K$ outcomes.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$-D_{KL}(p\Vert q) = \sum_x p(x)\log\frac{q(x)}{p(x)} \le \log\sum_x p(x)\frac{q(x)}{p(x)}$. For the second part, take $q$ uniform.

</details>

<details>
<summary><b>✅ Solution</b></summary>

By Jensen (log is concave): $-D_{KL} = \mathbb E_p\left[\log\frac qp\right]\le\log\mathbb E_p\left[\frac qp\right] = \log\sum_{x:p(x)>0}q(x)\le\log1=0$.
Equality in Jensen needs $q/p$ constant on the support of $p$, and summing to one forces $q=p$.
With $q=1/K$: $0\le D_{KL}(p\Vert u) = \sum p\log p + \log K = \log K - H(p)$, so $H(p)\le\log K$.

</details>

## Part C · Code

### Exercise 12 · Entropy, cross-entropy and KL in NumPy
*💻 Code · ★☆☆*

Implement `entropy(p, base=np.e)`, `cross_entropy(p, q)` and `kl(p, q)` for discrete distributions (1-D arrays). Use the
convention $0\log0=0$ (terms with $p_i=0$ contribute nothing) and normalise the inputs. Compare with `scipy.stats.entropy`.

In [ ]:
def entropy(p, base=np.e):
    return None
def cross_entropy(p, q):
    return None
def kl(p, q):
    return None

p_t = np.array([0.5, 0.3, 0.2, 0.0]); q_t = np.array([0.25, 0.25, 0.25, 0.25])

check('entropy (nats)', entropy(p_t), sp_entropy(p_t))
check('entropy (bits)', entropy(p_t, base=2), sp_entropy(p_t, base=2))
check('KL', kl(p_t, q_t), sp_entropy(p_t, q_t))
check('H(p,q) = H(p) + KL', cross_entropy(p_t, q_t), sp_entropy(p_t) + sp_entropy(p_t, q_t))

<details>
<summary><b>💡 Hint</b></summary>

Mask: `m = p > 0`, then `-np.sum(p[m] * np.log(p[m]))`. Divide by `np.log(base)` for other bases.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The mask implements $0\log0=0$ (the limit $\lim_{p\to0}p\log p=0$). If $q_i=0$ where $p_i>0$, KL and cross-entropy are $+\infty$, which is why models are made to output strictly positive probabilities.

```python
def entropy(p, base=np.e):
    p = np.asarray(p, float); p = p / p.sum(); m = p > 0
    return -np.sum(p[m] * np.log(p[m])) / np.log(base)
def cross_entropy(p, q):
    p = np.asarray(p, float) / np.sum(p); q = np.asarray(q, float) / np.sum(q); m = p > 0
    return -np.sum(p[m] * np.log(q[m]))
def kl(p, q):
    p = np.asarray(p, float) / np.sum(p); q = np.asarray(q, float) / np.sum(q); m = p > 0
    return np.sum(p[m] * np.log(p[m] / q[m]))

p_t = np.array([0.5, 0.3, 0.2, 0.0]); q_t = np.array([0.25, 0.25, 0.25, 0.25])
```

</details>

### Exercise 13 · Stable cross-entropy from logits
*💻 Code · ★★☆*

Implement `ce_from_logits(Z, y)`: the mean cross-entropy of softmax probabilities given raw scores `Z` (shape `(N, K)`) and integer
labels `y`, using the log-sum-exp trick $\log\sum_k e^{z_k} = m + \log\sum_k e^{z_k-m}$ with $m=\max_k z_k$. It must not overflow
for logits of size 1000. Compare with `sklearn.metrics.log_loss` on moderate logits.

In [ ]:
def ce_from_logits(Z, y):
    return None

Z_t = rng.normal(scale=3, size=(20, 4)); y_t = rng.integers(0, 4, 20)
Z_big = np.array([[1000., 0., -1000.]]); y_big = np.array([1])

_P = np.exp(Z_t - Z_t.max(1, keepdims=True)); _P /= _P.sum(1, keepdims=True)
check('CE vs sklearn', ce_from_logits(Z_t, y_t), log_loss(y_t, _P, labels=[0, 1, 2, 3]))
check('no overflow on huge logits', ce_from_logits(Z_big, y_big), 1000.0)

<details>
<summary><b>💡 Hint</b></summary>

$-\log\mathrm{softmax}(z)_y = \mathrm{LSE}(z) - z_y$. Use `Z[np.arange(N), y]` to pick the true-class logits.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Computing `log(softmax(z))` naively gives `exp(1000) = inf` or `log(0) = -inf`. Working in log-space via LSE is what `CrossEntropyLoss` in deep-learning frameworks does internally (fused log-softmax + NLL).

```python
def ce_from_logits(Z, y):
    m = Z.max(axis=1, keepdims=True)
    lse = m[:, 0] + np.log(np.exp(Z - m).sum(axis=1))
    return np.mean(lse - Z[np.arange(len(y)), y])

Z_t = rng.normal(scale=3, size=(20, 4)); y_t = rng.integers(0, 4, 20)
Z_big = np.array([[1000., 0., -1000.]]); y_big = np.array([1])
```

</details>

### Exercise 14 · Best split by information gain
*💻 Code · ★★☆*

Implement `best_split(X, y)` that tries every feature and every threshold (midpoints between sorted unique values) and returns
`(feature, threshold, gain)` maximising the information gain in bits. Run it on the iris data and compare the gain with the root
split of `DecisionTreeClassifier(max_depth=1, criterion='entropy')` (which uses $\log_2$).

In [ ]:
def best_split(X, y):
    # TODO: return (feature, threshold, information gain in bits)
    return None

X_iris, y_iris = load_iris(return_X_y=True)
split = best_split(X_iris, y_iris)

_t = DecisionTreeClassifier(max_depth=1, criterion='entropy').fit(X_iris, y_iris).tree_
_n = _t.n_node_samples
check('best information gain', None if split is None else split[2], _t.impurity[0] - (_n[1] * _t.impurity[1] + _n[2] * _t.impurity[2]) / _n[0])

<details>
<summary><b>💡 Hint</b></summary>

Entropy of labels: `p = np.bincount(y) / len(y)`, drop zeros. For each threshold, `left = X[:, j] <= t`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The best gain is $\log_2 3 - \frac{100}{150}\cdot 1 \approx 1.585-0.667 = 0.918$ bits: a petal feature separates setosa perfectly (left child pure, right child 50/50 between the other two classes). Petal length and petal width tie, so the chosen feature may differ from sklearn's.

```python
def _H(y):
    p = np.bincount(y) / len(y); p = p[p > 0]
    return -np.sum(p * np.log2(p))

def best_split(X, y):
    best = (None, None, -1.0); H0 = _H(y)
    for j in range(X.shape[1]):
        v = np.unique(X[:, j])
        for t in (v[:-1] + v[1:]) / 2:
            left = X[:, j] <= t
            g = H0 - left.mean() * _H(y[left]) - (~left).mean() * _H(y[~left])
            if g > best[2]:
                best = (j, t, g)
    return best

X_iris, y_iris = load_iris(return_X_y=True)
split = best_split(X_iris, y_iris)
print(split)
```

</details>

### Exercise 15 · Mutual information of discrete features
*💻 Code · ★★☆*

Implement `mutual_info(x, y)` (in nats) for two integer-coded arrays via the empirical joint table:
$I = \sum_{a,b}\hat p(a,b)\log\frac{\hat p(a,b)}{\hat p(a)\hat p(b)}$. Discretise each iris feature into 5 equal-width bins and rank
the features by MI with the class label. Compare with `sklearn.metrics.mutual_info_score`.

In [ ]:
def mutual_info(x, y):
    return None

X_bin = np.stack([np.digitize(c, np.linspace(c.min(), c.max(), 6)[1:-1]) for c in X_iris.T], axis=1)
mi_features = None   # list of 4 MI values (nats), one per iris feature

check('MI per feature', mi_features, [mutual_info_score(X_bin[:, j], y_iris) for j in range(4)])

<details>
<summary><b>💡 Hint</b></summary>

Build the joint counts with `np.add.at(C, (x, y), 1)` or `np.histogram2d`; skip cells with zero count.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The petal features have the highest MI (about $0.9$ nats, out of a maximum of $H(Y)=\ln 3\approx1.10$), sepal width the lowest (about $0.27$), matching what trees and scatter plots show. MI depends on the binning: more bins inflate MI estimates on small samples.

```python
def mutual_info(x, y):
    xs, xi = np.unique(x, return_inverse=True); ys, yi = np.unique(y, return_inverse=True)
    C = np.zeros((len(xs), len(ys))); np.add.at(C, (xi, yi), 1)
    P = C / C.sum(); px = P.sum(1, keepdims=True); py = P.sum(0, keepdims=True)
    m = P > 0
    return np.sum(P[m] * np.log(P[m] / (px @ py)[m]))

X_bin = np.stack([np.digitize(c, np.linspace(c.min(), c.max(), 6)[1:-1]) for c in X_iris.T], axis=1)
mi_features = [mutual_info(X_bin[:, j], y_iris) for j in range(4)]
print(np.round(mi_features, 3))
```

</details>

### Exercise 16 · Jensen–Shannon divergence
*💻 Code · ★★☆*

Implement `js(p, q)` $= \tfrac12D_{KL}(p\Vert m)+\tfrac12D_{KL}(q\Vert m)$ with $m=\tfrac12(p+q)$, in nats, reusing `kl`.
Verify on random distributions that it is symmetric, bounded by $\ln 2$, finite even when supports differ, and that
`scipy.spatial.distance.jensenshannon` returns $\sqrt{\text{JS}}$.

In [ ]:
def js(p, q):
    return None

p_js = rng.dirichlet(np.ones(5)); q_js = rng.dirichlet(np.ones(5))

check('JS = scipy distance squared', js(p_js, q_js), jensenshannon(p_js, q_js) ** 2)
check('symmetric', None if js(p_js, q_js) is None else np.isclose(js(p_js, q_js), js(q_js, p_js)), True)
check('disjoint supports give ln 2', js([1, 0], [0, 1]), np.log(2))

<details>
<summary><b>💡 Hint</b></summary>

Because $m>0$ wherever $p>0$ or $q>0$, the KL terms are always finite.

</details>

<details>
<summary><b>✅ Solution</b></summary>

JS is symmetric, bounded in $[0,\ln2]$, and $\sqrt{\text{JS}}$ is a true metric. In the original GAN analysis the optimal discriminator turns the generator objective into $2\,\text{JS}(p_{data}\Vert p_g)-\ln4$. Since JS saturates at $\ln2$ for disjoint supports, the gradients vanish, which motivated Wasserstein GANs.

```python
def js(p, q):
    p = np.asarray(p, float) / np.sum(p); q = np.asarray(q, float) / np.sum(q)
    m = (p + q) / 2
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

p_js = rng.dirichlet(np.ones(5)); q_js = rng.dirichlet(np.ones(5))
```

</details>

### Exercise 17 · Monte Carlo KL estimate
*💻 Code · ★★★*

Estimate $D_{KL}(\mathcal N(0,1)\Vert\mathcal N(1,4))$ by Monte Carlo: sample $x_i\sim p$ and average $\log p(x_i)-\log q(x_i)$.
Use $n=200\,000$ samples and store the estimate in `kl_mc` and its standard error in `kl_se`. Compare with the closed form from
Exercise 10. This is how the ELBO's KL term is estimated when no closed form exists.

In [ ]:
kl_mc = None
kl_se = None

check('MC estimate', kl_mc, np.log(2) + 2 / 8 - 0.5, tol=0.01)
check('standard error small', None if kl_se is None else 0 < kl_se < 0.005, True)

<details>
<summary><b>💡 Hint</b></summary>

Write the Gaussian log-density yourself or use `scipy.stats.norm(mu, sigma).logpdf`. Standard error = sample std / $\sqrt n$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The estimate is unbiased with error $\approx$ std$/\sqrt n$. Individual terms $\log p-\log q$ can be negative, only their mean is guaranteed $\ge 0$. VAEs use a single such sample per data point (with the reparameterisation trick) when the KL has no closed form.

```python
from scipy import stats
xs = np.random.default_rng(5).normal(0, 1, 200_000)
terms = stats.norm(0, 1).logpdf(xs) - stats.norm(1, 2).logpdf(xs)
kl_mc = terms.mean(); kl_se = terms.std() / np.sqrt(len(terms))
print(kl_mc, '+/-', kl_se)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Information Theory](Information%20Theory.md).*